# §1 Introduction

### 1.1 Aims

This report aims to investigate N-body gravitational simulations in relation to virialisation. Virialisation is the process where a gravitationally bound system attains a steady state where

$$
2T + U=0, \tag{1}
\\
\\
\frac{2T}{|U|} = 1. 
$$
With T being the total kinetic energy and U as the total gravitational potential energy **[1]**. This is a key process in physics, as it explains how dark matter halos ceased gravitational collapse and establish the underlying potential structure that enables galaxy formation **[2]**. Consequently, it is important to develop algorithms that accurately reproduce the Virial theorem when performing cosmological simulations.

In this report the time complexity of the pairwise, Barnes-Hut (BH) and fast multipole method (FMM) algorithms have been investigated for up to $10^{3}$ particles in regimes of $r_{virial} = 10^{-3}, 10^{0}, 10^{3}$. The physical accuracy of these systems was probed via observing momentum and energy conservation in these same regimes.

### 1.2 History

Computers became powerful enough in the mid 1980s for cosmologists to start running N-body gravity simulations. One of the best known by G. Efstathiou and M. Davis successfully used a particle and mesh-based algorithm to simulate ~32,000 particles **[3]**. The accuracy and scale of the programs has steadily increased over time and modern simulations like the 2022 MillenniumTNG Project simulated galactic stuctures in volumes of $(740Mpc)^{3}$ **[4]**.

Both of these projects were written in machine code or highly optimised C and were run on state-of-the-art supercomputers. By contrast, this project was written in Python on a personal laptop, and therefore has a smaller scope and much lower N.

# §2 Method and Algorithms

This simulation models all particles as having identical mass ($m=1$) as a simplification of the stellar mass spectrum. This is a reasonable approximation for studying bulk gravitational dynamics, since most of the stellar mass in typical galaxies is concentrated within sub-solar masses (roughly $0.1–1M_{\odot}$)
 and equal-mass models capture the leading-order behaviour of large-N self-gravitating systems.

The value of $G = 1$. This rescales the dynamical timescale of the system to order unity.
$$
t_{dyn} = \sqrt{\frac{1}{G\rho}} \tag{2}
$$ 
where $\rho$ is the density. This enables a convenient choice of $dt = 10^{-3} - 10^{-2}$.


All the algorithms aim to simulate the following acceleration 
$$
\vec{a_{ij}} = \frac{-Gm_{i}m_{j}}{(r_{ij}^{2} + \epsilon_{ij}^{2})} \vec{\hat{r_{ij}}} \tag{3}
$$
where the symbols have their usual meanings and $\epsilon$ is the softening factor. The softening only exists to remove divisions by zero for particles in the same place.









### 2.1 Simulation Setup

The system is represented by an "NBodySystem" class, initialised with arrays of particle positions and velocities. These are stored internally as (N,3) NumPy arrays, where N is the number of particles. The class exposes these as ".positions" and ".velocities", and defines ".N" as the total particle count. Validation checks are implemented to ensure the input arrays have consistent shapes and the correct dimensionality.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

class NBodySystem:
    def __init__(self, positions, velocities):

        ''' Positions and velocities are N by 3 numpy arrays.'''

        if positions.shape[0] != velocities.shape[0]:
            raise ValueError(
                f"\n*** Different numbers of particle for position and velocity***")
        
        if positions.shape[1] != 3 or velocities.shape[1] != 3:
            raise ValueError(f"\n*** Positions or velocities are not 3 dimensional***")

        self.positions = np.array(positions, dtype=float)      
        self.velocities = np.array(velocities, dtype=float)  
        self.N = len(self.positions)


The simulation is carried out by the "simulate()" function, which advances the system forward in time while recording its full trajectory. At each step, the current positions and velocities are stored, after which the system state is updated in place via the chosen integration scheme.

A key feature is the clear separation between the system state, the time-stepping method ("step_function"), and the acceleration calculation ("acceleration_function"). This modular structure allows different integrators or acceleration solvers to be swapped in and out without modifying the core simulation loop, making the framework flexible.

The trajectory is stored in memory and animated later. This allows for animations without lag and simplifies investigations of how system properties, such as total energy, evolve with time.

The design feature of "**acceleration_kawargs" with the wrapper function "acceleration_function_with_arguments" allows a uniform interface between acceleration functions while enabling pass through of extra arguments if required. Initially this was intended to be an "approximation_order" for the Fast Multipole Method Algorithm, however due to time constraints this was only implemented for the "threshold" for the Barnes-Hut algorithm (described later). 

In [2]:
def simulate(system,
            step_function,
            acceleration_function, 
            steps,
            dt=0.001,
            G=1, 
            softening=1e-5,
            **acceleration_kwargs):

    ''' Simulates and records trajectory of system.'''

    trajectory_positions = np.zeros((steps, system.N, 3))
    trajectory_velocities = np.zeros((steps, system.N, 3))

    def acceleration_function_with_arguments(sys):
        return acceleration_function(sys, G, softening, **acceleration_kwargs)

    for t in range(steps):
        trajectory_positions[t] = system.positions
        trajectory_velocities[t] = system.velocities

        step_function(system, acceleration_function_with_arguments, dt)
    
    return trajectory_positions, trajectory_velocities


### 2.2 Pairwise Algorithm

This is the simplest acceleration calculation algorithm, and operates by looping over every particle and calculating the force on that particle caused by all other particles.
The pseudocode is as follows:


<img src="pseudocode/pairwise.jpg" style="width:70%; display:block; margin:auto;">

**Fig. 1:** Pseudocode describing the pairwise N-body algorithm. Line 5 removes unphysical self-gravitation for point masses. The nested for-loop implies an $O(N^{2})$ time complexity. Newton's third law has not been exploited as an optimisation.


To implement this in Python, NumPy broadcasting was exploited in the "gravitational_vectorised_acceleration()" function. In the code below note the "dx = positions[:, np.newaxis, :] - positions[np.newaxis, :, :]" line. "positions[:, np.newaxis, :]" is an (N, 1, 3) array and positions[np.newaxis, :, :] is a (1, N, 3) array. As these shapes are compatible (though not the same) the result of the calculation is broadcast into the "dx" array of shape (N, N, 3) where "dx[i, j] = positions[i] - positions[j]"

Mathematically, this is equivalent to the following.

$$
\mathbf{dx} = \mathbf{1}\mathbf{x}^{T} - \mathbf{x}\mathbf{1}^{T} \tag{4},
\\
\text{where} \quad \mathbf{x} = \begin{pmatrix}
\vec{x_0}\\ \vec{x_1}\\ \vec{x_2}\\. \\ . \\. 
\end{pmatrix}
\\
\\
\text{and} \quad \mathbf{1} = \begin{pmatrix}
\vec{1}\\ \vec{1}\\ \vec{1}\\ . \\ . \\. 
\end{pmatrix}.
$$

All such operations are executed internally by NumPy in optimised C code, avoiding explicit Python loops and reducing overhead. As a result, this vectorised implementation is approximately 320 times faster than a naive Python loop-based approach (which can be found in "accelerations.py"). 

Self-interactions are removed by setting the diagonal elements of the inverse-distance term to zero via "np.fill_diagonal(inverse_distance_cubed, 0)", ensuring that particles do not exert forces on themselves.

Finally, the total acceleration on each particle is obtained by summing over all pairwise contributions, returning the expected array of shape (N, 3).

In [3]:
def gravitational_vectorised_acceleration(system, G=1, softening=1e-5):

    ''' Naive approach with Numpy broadcasting.'''
    # ~320 times faster than Numpy loops.

    positions = system.positions

    # Numpy broadcasting instead of looping over index.
    dx = positions[:, np.newaxis, :] - positions[np.newaxis, :, :]   

    dist_sq = np.sum(dx**2, axis=2) + softening**2    
    inverse_distance_cubed = 1.0 / (dist_sq * np.sqrt(dist_sq))

    np.fill_diagonal(inverse_distance_cubed, 0)

    forces_div_masses = -G * dx * inverse_distance_cubed[:, :, np.newaxis] 
    acceleration = np.sum(forces_div_masses, axis=1)

    return acceleration


### 2.3 Barnes-Hut (BH) Algorithm

The Barnes–Hut algorithm was introduced in 1986 by Josh Barnes and Piet Hut [5]. It builds on earlier hierarchical methods for computing gravitational interactions, such as Appel's algorithm, which organises particles into a tree structure to compute a potential surface which is then differentiated to find the force.[6]. (It was later found that under some conditions Appel's algorithm is actually O(N) [7].) The key innovation of BH is that it includes a $\theta$ parameter that allows easy tuning of accuracy vs computational overhead. 

The Barnes–Hut algorithm operates by partitioning a system of N particles into hierarchical groupings stored in an octree. Each node in the tree corresponds to a region of physical space. If a particle is in a node which already contains a particle, the node is recursively subdivided until each subdivision contains at most one particle. The tree consists of two node types: internal (root) nodes, which contain child nodes and no particles, and external (leaf) nodes, which have no children but may contain a particle. The centre of mass (COM) and total mass of the particles in a node are calculated as the tree is constructed. The tree is completely rebuilt every time step.

To compute the acceleration on a single particle the tree is traversed starting from the root. If a node is sufficiently far from the particle, it is approximated as being a point mass positioned at the COM and with total mass of the node. This is a dipole approximation. If the node is sufficiently close to the particle the process is repeated for the node's children. 

A node is considered sufficiently distant if the ratio of its width to the separation between its centre of mass and the particle of interest is less than a tunable threshold $\theta$, i.e.

$$
\frac{s}{r} < \theta, \tag{5}
$$

where s is the node size and r is the distance to its centre of mass. $\theta$ affects the accuracy of the simulation, large values of $\theta \sim 1$ reduce runtime but also accuracy. If $\theta = 0$, no approximations are made and BH reduces to the pairwise algorithm.


<img src="pseudocode/Barnes-Hut create_node 1.png" style="width:70%; display:block; margin:auto;">

**Fig. 2:** Pseudocode for initialising a node. Note that "node centre" and "node COM" are vector quantities.

<img src="pseudocode/Barnes-hut insert 2.png" style="width:70%; display:block; margin:auto;">

**Fig. 3:** Pseudocode for inserting a particle into a node. First it checks to see if the particle is physically within the node. Then it updates the mass and COM of the node. After this it checks to see if the node already contains a particle. If so, 8 equal sized child nodes are produced and both particles are placed in them. This is recursively done untill every particle is in its own node. Note that root nodes only have children, they do not contain particles.

<img src="pseudocode/Barnes-Hut acceleration 3.png" style="width:70%; display:block; margin:auto;">

Fig. 4: Tree traversal for acceleration calculation. Self-interactions are excluded, leaf nodes are evaluated directly, and internal nodes are either approximated using their COM and mass or recursively opened depending on the $\theta$ criterion. Note that the leaf nodes are not approximated as their node properties are identical to the particle within (should they contain one).

<img src="pseudocode/Barnes-Hut system_acceleration 4.png" style="width:70%; display:block; margin:auto;">

Fig. 5: Calculation of particle accelerations. The first loop constructs the octree and inserts the particles. The second loop computes the acceleration of each particle via tree traversal. The acceleration $\vec{a_{particle}}$ is reset to zero at each timestep, as forces are totally recomputed rather than updated from the previous step.

To estimate the time complexity, one must consider the depth of the tree. For a roughly uniform particle distribution, the mean separation scales as $\Delta x \sim L/N^{1/3}$, where $L$ is the system size. Since leaf nodes contain at most one particle, their width is $\sim \Delta x$. Each subdivision halves the cell size, so the mean tree depth is $\sim \log_{2}(N) / 3$
implying $O(\log N)$ work per traversal. Inserting N particles therefore requires $O(N\log (N))$ operations. 

The time complexity of the acceleration calculation can be understood by considering the work required for a single particle. For a homogeneous distribution, increasing the number of particles by a factor of eight corresponds to adding one additional level to the tree (it is like joining eight similar nodes together). This increases the number of node interactions by a constant amount (dependent on $\theta$, but independent of $N$). Thus, the work per particle scales as $O(\log N)$, implying a total cost of $O(N \log N)$ to compute all accelerations.

As both tree construction and force evaluation scale as $O(N \log N)$, the overall algorithm has the same complexity.


In Python it looks like the following.

In [ ]:
class OctTreeNode:
    def __init__(self, centre, half_size):
        self.centre = centre
        self.half_size = half_size

        self.children = [None for i in range(8)]
        self.particle_index = None

        self.mass = 0
        self.centre_of_mass = np.zeros(3)
        


def get_octant(node, position):

    ''' Finds the octant of a particle. Stores position in bits as {xyz}. '''

    centre = node.centre
    index = 0
    if position[0] > centre[0]:
        index |= 1

    if position[1] > centre[1]: 
        index |= 2

    if position[2] > centre[2]:
        index |= 4

    return index
    

def create_child(node, octant):

    ''' Creates new node from old node.'''

    offset = np.array([
    0.5 if octant & 1 else -0.5,
    0.5 if octant & 2 else -0.5,
    0.5 if octant & 4 else -0.5
])

    new_centre = node.centre + (offset * node.half_size)
    return OctTreeNode(new_centre, node.half_size/2)
    

def insert(node, system, index):
    particle_position = system.positions[index]

    # Empty leaf.

    if node.particle_index is None and node.children[0] is None:
        node.particle_index = index
        return

    if node.children[0] is None:
        for i in range(8):
            node.children[i] = create_child(node, i)

        # Puts any exising particle into a child node. This is the recursive part.

        if node.particle_index is not None:
            old_index = node.particle_index
            node.particle_index = None
            insert(node, system, old_index)

    octant = get_octant(node, particle_position)
    insert(node.children[octant], system, index)


def compute_mass_distribution(node, system):

    ''' Finds mass, centre of mass of a node. '''

    # Empty node.
    if node is None:
        return 0.0, np.zeros(3)
    
    # Leaf node with particle.
    if node.particle_index is not None:
        node.mass = 1.0
        node.centre_of_mass = system.positions[node.particle_index]
        return node.mass, node.centre_of_mass
    
    # Internal node
    total_mass = 0.0
    com = np.zeros(3)

    for child in node.children:
        if child is None:
            continue
        m, c = compute_mass_distribution(child, system)
        total_mass += m
        com += m * c

    if total_mass > 0:
        com /= total_mass

    node.mass = total_mass
    node.centre_of_mass = com

    return total_mass, com
    

def compute_gravitational_acceleration(node, system, i, theta, G, softening):

    ''' Uses Barnes-Hut to find the gravitational acceleration on particle i due to a node. '''

    if node is None or node.mass == 0:
        return np.zeros(3)

    pos_i = system.positions[i]

   
    if node.particle_index == i:
        return np.zeros(3)

    r = node.centre_of_mass - pos_i
    dist_sq = np.dot(r, r) + softening**2
    dist = np.sqrt(dist_sq)

    s = node.half_size * 2

    
    if (node.particle_index is not None) or (s / dist < theta):
        inv_dist3 = 1.0 / (dist_sq * dist)
        return G * r * inv_dist3

    
    acc = np.zeros(3)
    for child in node.children:
        if child is not None:
            acc += compute_gravitational_acceleration(child, system, i, theta, G, softening)

    return acc


Note that this implementation differs slightly from the pseudocode, as node masses and centres of mass are computed in a separate pass via "compute_mass_distribution()" after the tree has been constructed using "insert()". This separation of responsibilities simplifies debugging.

The functions "get_octant()" and "create_child()" encode the particle’s position relative to the node centre as a three-bit index. Bitwise OR operations are used to construct this index, allowing the correct child node to be selected without multiple nested conditional IF statements.

### 2.3 Fast Multipole Method (FMM) Algorithm
The Fast Multipole Method (FMM) was introduced in 1987 by Leslie Greengard and Vladimir Rokhlin [8]. Like the Barnes–Hut algorithm, it employs a hierarchical octree to organise particles, but extends this approach by allowing node-node interactions via multipole expansions. This is the most complicated of the three algorithms.

First the space is divided into an octree of depth $\log(N)$. This is a perfect tree - independent of particle distribution and every root node has eight children except leaf nodes. A parameter $P$ sets the order of the truncated expansions about each node’s centre. Each node carries two representations: a multipole expansion describing the contribution of masses within the node, and a local expansion representing the influence of distant masses. In addition, each node maintains an interaction list consisting of well-separated nodes at the same level.

An upward pass begins at the leaves, where multipole expansions are computed from the enclosed particles. These are then aggregated up the tree, with each parent forming its multipole expansion from those of its children, continuing recursively to the root.

A downward pass then propagates information from the root to the leaves. For each node, a local expansion is formed by combining contributions from the multipole expansions of nodes in its interaction list with the local expansion inherited from its parent. This process continues to the leaf nodes, where the expansions are used to evaluate forces.

Particle accelerations are found by performing particle - particle calculations for particles in the same or adjacent leaves. The internal and external potentials are then evaluated at the particle positions, and they are combined with the particle - particle calculations. 




# §-1 References and Acknowledgements

**[1]**: J.A. Peacock, *Cosmological Physics*, 1st Edition, pp. 554 (1999)

**[2]**: T. Padmanabhan, *Structure Formation in the Universe*, 1st Edition, pp. 391 (1993)

**[3]**: G. Efstathiou, M. Davis, C.S. Frenk, S.D.M White, *Numerical Techniques for Large Cosmological N-Body Simulations*, Astrophysical Journal Supplement Series, Volume 57, pp. 241-260 (1985)

**[4]**: R. Pakmor et al., *The MillenniumTNG Project: The Hydrodynamical Full Physics Simulation and a First Look at its Galaxy Clusters*, Monthly Notices of the Royal Astronomical Society (2023)

**[5]**: J.Barnes, P.Hut, *A Hierarchical O(N log N) Force-Calculation Algorithm*, Nature, Volume 324, Issue 6096, pp. 446-449 (1986)

**[6]**: A. Appel, *An Efficient Program for Many-Body Simulation*, SIAM Journal on Scientific and Statistical Computing, Volume 6, Issue 1, pp. 85-103 (1985)

**[7]**: K. Esselink, *The order of Appel's algorithm*, Information Processing Letters, Volume 41, Issue 3, pp. 141-147 (1992) 

**[8]**: L. Greengard, V. Rokhlin Jr., *A Fast Algorithm for Particle Simulations*, Journal of Computational Physics, Volume 73, Issue 2, pp. 325-348 (1987)